# QuickPay Fintech Pipeline

This notebook covers:
- Part 3: Reconciliation between ledger.csv and gateway.csv
- Part 4: JSON normalization of api_response_sample.json
- Dashboard data preparation for Part 5

## Part 3 — Reconciliation Workflow

### 3.1 Load the data

In [ ]:
import pandas as pd
import numpy as np
import json

ledger = pd.read_csv('../01_data/raw/ledger.csv')
gateway = pd.read_csv('../01_data/raw/gateway.csv')

print(f"Ledger shape: {ledger.shape}")
print(f"Gateway shape: {gateway.shape}")
print()
ledger.head()

In [ ]:
gateway.head()

### 3.2 Check for duplicates and nulls

In [ ]:
print("Ledger duplicates:", ledger['transaction_id'].duplicated().sum())
print("Gateway duplicates:", gateway['transaction_id'].duplicated().sum())
print()
print("Ledger nulls per column:")
print(ledger.isnull().sum())
print()
print("Gateway nulls per column:")
print(gateway.isnull().sum())

No duplicates and no nulls in either file. Both datasets are clean on the structural level.

### 3.3 Identify missing records

In [ ]:
ledger_ids = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

missing_in_gw_ids = ledger_ids - gateway_ids
missing_in_led_ids = gateway_ids - ledger_ids

print(f"Records in ledger but missing in gateway: {missing_in_gw_ids}")
print(f"Records in gateway but missing in ledger: {missing_in_led_ids}")

In [ ]:
missing_in_gateway = ledger[ledger['transaction_id'].isin(missing_in_gw_ids)]
missing_in_ledger = gateway[gateway['transaction_id'].isin(missing_in_led_ids)]

print("Missing in gateway:")
print(missing_in_gateway.to_string(index=False))
print()
print("Missing in ledger:")
print(missing_in_ledger.to_string(index=False))

missing_in_gateway.to_csv('../01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('../01_data/processed/missing_in_ledger.csv', index=False)

R004 and R010 exist in the ledger but have no matching gateway entry. R011 is in the gateway but not recorded in the ledger.

### 3.4 Amount mismatches

In [ ]:
common_ids = ledger_ids & gateway_ids

led_common = ledger[ledger['transaction_id'].isin(common_ids)].set_index('transaction_id')
gw_common = gateway[gateway['transaction_id'].isin(common_ids)].set_index('transaction_id')

amt_compare = pd.DataFrame({
    'ledger_amount': led_common['amount_usd'],
    'gateway_amount': gw_common['amount_usd']
})
amt_compare['difference'] = abs(amt_compare['ledger_amount'] - amt_compare['gateway_amount'])
amount_mismatches = amt_compare[amt_compare['difference'] > 0.01].reset_index()

print("Amount mismatches:")
print(amount_mismatches.to_string(index=False))

amount_mismatches.to_csv('../01_data/processed/amount_mismatches.csv', index=False)

R002 has a $50 discrepancy (ledger=850, gateway=900) and R008 has a $40 discrepancy (ledger=640, gateway=600).

### 3.5 Status mismatches

In [ ]:
stat_compare = pd.DataFrame({
    'ledger_status': led_common['status'],
    'gateway_status': gw_common['status']
})
status_mismatches = stat_compare[stat_compare['ledger_status'] != stat_compare['gateway_status']].reset_index()

print("Status mismatches:")
print(status_mismatches.to_string(index=False))

status_mismatches.to_csv('../01_data/processed/status_mismatches.csv', index=False)

R005 shows 'success' in the ledger but 'failed' in the gateway. This needs investigation — the payment may not have actually settled.

### 3.6 Build reconciliation report

In [ ]:
recon_rows = []
all_ids = sorted(ledger_ids | gateway_ids)

for tid in all_ids:
    row = {'transaction_id': tid}
    in_l = tid in ledger_ids
    in_g = tid in gateway_ids
    row['in_ledger'] = in_l
    row['in_gateway'] = in_g

    if in_l and not in_g:
        lr = ledger[ledger['transaction_id'] == tid].iloc[0]
        row['ledger_amount'] = lr['amount_usd']
        row['gateway_amount'] = np.nan
        row['ledger_status'] = lr['status']
        row['gateway_status'] = np.nan
        row['issue'] = 'missing_in_gateway'
    elif in_g and not in_l:
        gr = gateway[gateway['transaction_id'] == tid].iloc[0]
        row['ledger_amount'] = np.nan
        row['gateway_amount'] = gr['amount_usd']
        row['ledger_status'] = np.nan
        row['gateway_status'] = gr['status']
        row['issue'] = 'missing_in_ledger'
    else:
        lr = ledger[ledger['transaction_id'] == tid].iloc[0]
        gr = gateway[gateway['transaction_id'] == tid].iloc[0]
        row['ledger_amount'] = lr['amount_usd']
        row['gateway_amount'] = gr['amount_usd']
        row['ledger_status'] = lr['status']
        row['gateway_status'] = gr['status']
        issues = []
        if abs(lr['amount_usd'] - gr['amount_usd']) > 0.01:
            issues.append('amount_mismatch')
        if lr['status'] != gr['status']:
            issues.append('status_mismatch')
        row['issue'] = ';'.join(issues) if issues else 'matched'

    recon_rows.append(row)

recon_df = pd.DataFrame(recon_rows)
print(recon_df.to_string(index=False))

recon_df.to_csv('../01_data/processed/reconciliation_report.csv', index=False)

### 3.7 Summary metrics

In [ ]:
issue_count = len(recon_df[recon_df['issue'] != 'matched'])

risk_from_mismatches = amount_mismatches['difference'].sum()
risk_from_missing_gw = missing_in_gateway['amount_usd'].sum()
risk_from_missing_led = missing_in_ledger['amount_usd'].sum()
amount_at_risk = round(risk_from_mismatches + risk_from_missing_gw + risk_from_missing_led, 2)

metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "reconciliation_issue_count": int(issue_count),
    "ledger_total_amount": float(ledger['amount_usd'].sum()),
    "gateway_total_amount": float(gateway['amount_usd'].sum()),
    "amount_at_risk": float(amount_at_risk)
}

with open('summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

for k, v in metrics.items():
    print(f"{k}: {v}")

Out of 11 unique transaction IDs across both systems, 6 have reconciliation issues:
- 2 missing in gateway (R004, R010)
- 1 missing in ledger (R011)
- 2 amount mismatches (R002, R008)
- 1 status mismatch (R005)

Total amount at risk is $6,490.

---
## Part 4 — JSON Normalization

### 4.1 Load and inspect the JSON

In [ ]:
with open('../01_data/raw/api_response_sample.json') as f:
    api_data = json.load(f)

print("Top-level keys:", list(api_data.keys()))
print("Number of batches:", len(api_data['batches']))
print()

# look at the structure
for b in api_data['batches']:
    print(f"Batch {b['batch_id']}: merchant={b['merchant']['merchant_name']}, settlements={len(b['settlements'])}")

### 4.2 Flatten into tabular form

In [ ]:
rows = []
for batch in api_data['batches']:
    for s in batch['settlements']:
        rows.append({
            'batch_id': batch['batch_id'],
            'merchant_id': batch['merchant']['merchant_id'],
            'merchant_name': batch['merchant']['merchant_name'],
            'region': batch['merchant']['region'],
            'settlement_id': s['settlement_id'],
            'amount_usd': s['amount_usd'],
            'status': s['status'],
            'processed_at': s['processed_at'],
            'bank_name': s['bank']['name'],
            'bank_country': s['bank']['country']
        })

api_df = pd.DataFrame(rows)
print(f"Flattened rows: {len(api_df)}")
api_df

### 4.3 Clean columns and convert dates

In [ ]:
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])

print("Column types:")
print(api_df.dtypes)
print()
print("Date range:", api_df['processed_at'].min(), "to", api_df['processed_at'].max())

### 4.4 Save normalized output

In [ ]:
api_df.to_csv('../01_data/processed/api_normalized.csv', index=False)
print("api_normalized.csv saved with", len(api_df), "rows")
api_df

The nested JSON had 2 batches with 3 settlements each. After flattening, we have 6 rows with clean column names. The bank object was unpacked into bank_name and bank_country. The processed_at timestamp was converted to proper datetime.

---
## Dashboard Data Preparation

Generating the summary CSVs that will feed into the Looker Studio dashboard.

In [ ]:
txn = pd.read_csv('../01_data/processed/cleaned_transactions.csv')
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])

# daily summary
daily = txn.groupby('transaction_date').agg(
    total_txns=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[txn.loc[x.index, 'status'] == 'captured'].sum()),
    successful_txns=('status', lambda x: (x == 'captured').sum()),
    failed_txns=('status', lambda x: (x == 'failed').sum()),
    chargeback_txns=('status', lambda x: (x == 'chargeback').sum())
).reset_index()
daily['success_rate_pct'] = (daily['successful_txns'] / daily['total_txns'] * 100).round(1)
daily.to_csv('../01_data/processed/daily_summary.csv', index=False)
print("daily_summary.csv saved")
daily

In [ ]:
# payment method breakdown
pm = txn.groupby('payment_method').agg(
    total_txns=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[txn.loc[x.index, 'status'] == 'captured'].sum()),
    avg_risk_score=('risk_score', 'mean')
).reset_index()
pm['avg_risk_score'] = pm['avg_risk_score'].round(1)
pm.to_csv('../01_data/processed/payment_method_breakdown.csv', index=False)
print("payment_method_breakdown.csv saved")
pm

In [ ]:
# region breakdown
rb = txn.groupby('gateway_region').agg(
    total_txns=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[txn.loc[x.index, 'status'] == 'captured'].sum()),
    avg_risk_score=('risk_score', 'mean'),
    high_value_count=('high_value_flag', 'sum'),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()
rb['avg_risk_score'] = rb['avg_risk_score'].round(1)
rb.to_csv('../01_data/processed/region_breakdown.csv', index=False)
print("region_breakdown.csv saved")
rb

In [ ]:
# merchant performance summary
mp = txn.groupby(['merchant_id', 'merchant_name', 'gateway_region']).agg(
    total_txns=('transaction_id', 'count'),
    total_gmv_usd=('amount_usd', 'sum'),
    captured_gmv_usd=('amount_usd', lambda x: x[txn.loc[x.index, 'status'] == 'captured'].sum()),
    failed_txns=('status', lambda x: (x == 'failed').sum()),
    chargeback_txns=('status', lambda x: (x == 'chargeback').sum()),
    avg_risk_score=('risk_score', 'mean'),
    high_value_count=('high_value_flag', 'sum'),
    high_risk_count=('high_risk_flag', 'sum')
).reset_index()
mp['success_rate_pct'] = (mp['captured_gmv_usd'] / mp['total_gmv_usd'] * 100).round(1)
mp['chargeback_ratio_pct'] = (mp['chargeback_txns'] / mp['total_txns'] * 100).round(2)
mp['avg_risk_score'] = mp['avg_risk_score'].round(1)
mp.to_csv('../01_data/processed/merchant_performance_summary.csv', index=False)
print("merchant_performance_summary.csv saved")
mp

All dashboard source files have been generated in `01_data/processed/`. These will be uploaded to Looker Studio as data sources.